In [18]:
import pandas 

In [19]:
csv_file = '../../data/raw/final_merged_data.csv'
raw_df = pandas.read_csv(csv_file)

# Preprocess

In [20]:
relevant_events = [
    'Stimuli Presentation',
    'Memory Quiz Response',
    'Stimuli Placed',
    'Memory Placement confidence',
    'Memory Quiz2 Response',
    'Memory Quiz2 confidence'
]
df_events = raw_df[raw_df['V3'].str.contains('|'.join(relevant_events), na=False)].copy()


In [51]:
trials = []

current_trial = {}

for _, row in df_events.iterrows():
    event = row['V3']
    
    if 'Stimuli Presentation' in event:
        # Start a new trial
        if current_trial:  # save previous trial
            trials.append(current_trial)
        current_trial = {
            'Participant': row['Participant'],
            'V1': row['V1'],
            'stimuli': row['V4'],
            'stimuli_cat': row['V6']
        }
        
        
    elif 'Memory Quiz Response' in event:
        current_trial['response'] = row['V4']
        
    elif 'Stimuli Placed' in event:
        # Extract distance if present in V9
        try:
            dist_str = row['V10']  # e.g., "Distance : 222.2831"
            distance = float(dist_str.split(':')[1].strip())
        except:
            distance = None
        current_trial['placement_distance'] = distance
        
    elif 'Memory Placement confidence' in event:
        current_trial['placement_cr'] = row['V4']
        
    elif 'Memory Quiz2 Response' in event:
        current_trial['seen_when'] = row['V4']
        
    elif 'Memory Quiz2 confidence' in event:
        current_trial['seen_when_cr'] = row['V4']

# Add last trial
if current_trial:
    trials.append(current_trial)


In [54]:
# Function to split and assign columns
def split_stimulus(row):
    parts = row['stimuli'].split('_')
    num_parts = len(parts)
    if num_parts >= 8:  # long stimuli
        return pd.Series({
            'stimuli_cat': parts[0],
            'list_no': parts[1],
            'encoding_no': parts[2],
            'stimuli_no': parts[3],
            'relevance': parts[4],
            'catego': parts[5],
            'old_new': parts[7]  # skip unknown1 (parts[6])
        })
    else:  # short stimuli
        return pd.Series({
            'stimuli_cat': parts[0],
            'list_no': None,
            'encoding_no': None,
            'stimuli_no': parts[1] if len(parts) > 1 else None,
            'relevance': None,
            'catego': None,
            'old_new': parts[2] if len(parts) > 2 else None
        })

In [60]:
trials_df = pandas.DataFrame(trials)
# Apply function to each row
split_cols = trials_df.apply(split_stimulus, axis=1)

# Merge back to trials_df
trials_df = pd.concat([trials_df, split_cols], axis=1)


trials_df = trials_df.drop(columns=['stimuli_cat', 'stimuli', 'V1']) 

# Strip the P of Participant column and convert to int
trials_df['Participant'] = trials_df['Participant'].str.lstrip('P').astype(int)
trials_df['placement_distance'] = trials_df['placement_distance'].astype(float)
trials_df['placement_cr'] = trials_df['placement_cr'].astype(int)
trials_df['seen_when_cr'] = trials_df['seen_when_cr'].astype(int)

# One-hot encode categorical columns
categorical_cols = ['response', 'seen_when', 'relevance', 'old_new', 'catego', 'encoding_no']
trials_df = pandas.get_dummies(trials_df, columns=categorical_cols)
# drop categorical columns
print(trials_df)

      Participant  placement_distance  placement_cr  seen_when_cr list_no  \
0              10            222.2831            23            37       2   
1              10           1118.3110           100           100    None   
2              10            543.8705             4            46       6   
3              10            754.4911            78            78       2   
4              10            745.0938            32            21       8   
...           ...                 ...           ...           ...     ...   
2235           51            129.5064            50            50       5   
2236           51            341.9683            10            50       3   
2237           51           1041.6010            50            50       3   
2238           51            406.5724            10           100      12   
2239           51           1198.5510           100           100    None   

     stimuli_no  response_not_seen  response_seen  seen_when_never  \
0    

In [61]:
# Check for NaNs
print(trials_df.isna().sum())

Participant              0
placement_distance       0
placement_cr             0
seen_when_cr             0
list_no                560
stimuli_no               0
response_not_seen        0
response_seen            0
seen_when_never          0
seen_when_one_week       0
seen_when_two_weeks      0
seen_when_yesterday      0
relevance_nr             0
relevance_r              0
old_new_memo             0
old_new_new              0
old_new_old              0
catego_h                 0
catego_o                 0
encoding_no_e1           0
encoding_no_e2           0
encoding_no_e3           0
dtype: int64


In [62]:
print(trials_df.describe(include='all'))

        Participant  placement_distance  placement_cr  seen_when_cr list_no  \
count   2240.000000         2240.000000   2240.000000   2240.000000    1680   
unique          NaN                 NaN           NaN           NaN      12   
top             NaN                 NaN           NaN           NaN       4   
freq            NaN                 NaN           NaN           NaN     210   
mean      29.571429          726.461841     54.597321     69.583036     NaN   
std       12.805282          462.201373     33.624618     29.535246     NaN   
min       10.000000            0.656116      0.000000      0.000000     NaN   
25%       18.000000          333.340800     25.000000     50.000000     NaN   
50%       27.000000          615.610000     50.000000     77.000000     NaN   
75%       42.000000         1153.416000     90.000000    100.000000     NaN   
max       51.000000         2008.334000    100.000000    100.000000     NaN   

       stimuli_no response_not_seen response_seen s

In [63]:
# Save preprocessed data
output_file = '../../data/preprocessed/memory_task_data_preprocessed.csv'
trials_df.to_csv(output_file, index=False)